# Example matching TMT

In this notebook, we introduce an illustrative example for the induced matching computation by using merge trees.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import scipy.spatial.distance as dist
import itertools

import tdqual.topological_data_quality_0 as tdqual

import os 
plots_dir = os.path.join("plots", "example_TMT")
os.makedirs(plots_dir, exist_ok=True)

In [ ]:
# pip install gudhi
import gudhi

In [ ]:
from sklearn.mixture import GaussianMixture

# 1. Define the parameters
n_samples = 12
n_components = 3

# Define centers to ensure they are clearly separated
# (x, y) coordinates for the 5 clusters
centers = np.array([
    [0, 0], [10, 5], [0, 10], [3,5]
])


weights = np.array([0.3, 0.3, 0.3, 0.1])

# 2. Initialize the GMM
gmm = GaussianMixture(n_components=n_components, random_state=42)

# Manually set the parameters to force the desired distribution
gmm.means_ = centers
gmm.weights_ = weights
# Set small covariances so clusters remain tight and separated
covariances = [1,3,3,4]
gmm.covariances_ = np.array([np.eye(2) * c for c in covariances])
# Required for initialization
gmm.precisions_cholesky_ = np.linalg.cholesky(np.linalg.inv(gmm.covariances_))

# 3. Sample from the model
Z, y = gmm.sample(n_samples)

Take a sample $X$ from $Z$ and plot the result.

In [ ]:
size_sample_X = 6
# Take the sample X
rng = np.random.default_rng(seed=4)
X_indices = rng.choice(Z.shape[0],size_sample_X, replace=False)
X =Z[X_indices]
X_compl_indices = list(set(range(X.shape[0]))-set(X_indices))
Z = np.vstack((Z[X_indices], Z[X_compl_indices]))
# Plot point cloud
fig, ax = plt.subplots(ncols=1, figsize=(4,4))
ax.scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=3, label="X")
ax.scatter(Z[:,0], Z[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="X", zorder=2, label="Z")
plt.legend(loc="upper right")
plt.savefig(os.path.join(plots_dir,"ex_TMT_sample.png"))

In [ ]:
### Geometric Matching 
def compute_components(edgelist, num_points):
    components = np.array(range(num_points))
    for edge in edgelist:
        max_idx = np.max(components[edge])
        min_idx = np.min(components[edge])
        indices = np.nonzero(components == components[max_idx])[0]
        components[indices]=np.ones(len(indices))*components[min_idx]
    
    return components

def plot_Vietoris_Rips_subset(Z, X_indices, filt_val, ax, labels=False, fontsize=15, vertex_size=150):
    X = Z[X_indices]
    # Plot point cloud
    if labels:
        ax.scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=vertex_size, marker="o", zorder=2)
        ax.scatter(Z[:,0], Z[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=vertex_size, marker="o", zorder=1)
    else:
        ax.scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=2)
        ax.scatter(Z[:,0], Z[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="o", zorder=1)
    # Plot simplicial complex 
    rips_complex = gudhi.RipsComplex(points=Z, max_edge_length=filt_val)
    simplex_tree = rips_complex.create_simplex_tree(max_dimension=1)
    simplex_tree.expansion(2)
    edgelist = []
    for filtered_value in simplex_tree.get_filtration():
        simplex = filtered_value[0]
        if len(simplex)==2:
            edgelist.append(simplex)
            if len(np.intersect1d(simplex, X_indices))==2:
                ax.plot(Z[simplex][:,0], Z[simplex][:,1], linewidth=2, c=mpl.colormaps["RdBu"](0.3/1.3), zorder=0.5)
            else:
                ax.plot(Z[simplex][:,0], Z[simplex][:,1], linewidth=2, c=mpl.colormaps["RdBu"](1/1.3), zorder=0.5)
        # if len(simplex)==3:
        #     ax.add_patch(mpl.patches.Polygon(Z[simplex], closed=True, facecolor="black", alpha=0.3, zorder=0.2))
    ax.set_aspect("equal")
    # Adjust margins
    xscale = ax.get_xlim()[1]-ax.get_xlim()[0]
    yscale = ax.get_ylim()[1]-ax.get_ylim()[0]
    xlim = ax.get_xlim()
    xlim = (xlim[0]-xscale*0.1, xlim[1]+xscale*0.1)
    ax.set_xlim(xlim)
    ylim = ax.get_ylim()
    ylim = (ylim[0]-yscale*0.1, ylim[1]+yscale*0.1)
    ax.set_ylim(ylim)
    # Plot labels
    if labels:
        components = compute_components(edgelist, Z.shape[0])
        # Point Labels 
        for i in range(Z.shape[0]):
            if components[i] < 10:
                ax.text(Z[i,0]-0.035*xscale, Z[i,1]-0.035*yscale, f"{components[i]}", fontsize=fontsize, color="white", fontweight="bold")
            else:
                ax.text(Z[i,0]-0.055*xscale, Z[i,1]-0.035*yscale, f"{components[i]}", fontsize=fontsize, color="white", fontweight="bold")
    # Finish with aspect details 
    # ax.set_xticks([])
    # ax.set_yticks([])
    ax.grid(True, color="gray", alpha=0.2)

In [ ]:
filtrations = [0,2,4,6, 8]
fig, ax = plt.subplots(nrows=2, ncols=len(filtrations), figsize=(3*len(filtrations),6))
for i, filt_val in enumerate(filtrations):
    plot_Vietoris_Rips_subset(Z, range(X.shape[0]), filt_val, ax[0, i], labels=True, fontsize=12,vertex_size=220)
    X = Z[:X.shape[0]]
    plot_Vietoris_Rips_subset(X, [], filt_val, ax[1, i], labels=True, fontsize=12,vertex_size=220)
    # Plot point cloud extra large
    ax[0,i].set_title(f"{filt_val:.1f}", fontsize=18)

ax[0,0].set_title("Dataset", fontsize=20)
ax[1,0].set_title("Subset", fontsize=20)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir,"VR_components.png"))

Next, we compute the minimum spanning grees for $X$ and $Z$:

In [ ]:
import scipy.spatial.distance as dist
Dist_X = dist.squareform(dist.pdist(X))
Dist_Z = dist.squareform(dist.pdist(Z))
filt_X, pairs_arr_X = tdqual.mst_edge_filtration(X) # MST(X)
filt_Z, pairs_arr_Z = tdqual.mst_edge_filtration(Z) # MST(Z)

In [ ]:
def plot_mst(points, mst_edges, ax, vertex_size=50, color="black"):
    ax.scatter(points[:,0], points[:,1], s=vertex_size, marker="o", zorder=2, color=color)
    for edge in mst_edges:
        ax.plot(points[edge][:,0], points[edge][:,1], linewidth=2, color=color, zorder=0.5)
    ax.set_aspect("equal")
    # Plot labels
    components = compute_components(mst_edges, points.shape[0])
    # Adjust margins
    xscale = ax.get_xlim()[1]-ax.get_xlim()[0]
    yscale = ax.get_ylim()[1]-ax.get_ylim()[0]
    xlim = ax.get_xlim()
    xlim = (xlim[0]-xscale*0.1, xlim[1]+xscale*0.1)
    ax.set_xlim(xlim)
    ylim = ax.get_ylim()
    ylim = (ylim[0]-yscale*0.1, ylim[1]+yscale*0.1)
    ax.set_ylim(ylim)
    # Finish with aspect details 
    # ax.set_xticks([])
    # ax.set_yticks([])
    ax.grid(True, color="gray", alpha=0.2)

In [ ]:
fig, ax = plt.subplots(ncols=3, figsize=(9,3))
plot_mst(X, pairs_arr_X, ax[0], color=mpl.colormaps["RdBu"](0.3/1.3))
plot_mst(Z, pairs_arr_Z, ax[1], color=mpl.colormaps["RdBu"](1/1.3))
plot_mst(Z, pairs_arr_Z, ax[2], color=mpl.colormaps["RdBu"](1/1.3))
plot_mst(X, pairs_arr_X, ax[2], color=mpl.colormaps["RdBu"](0.3/1.3))
ax[0].set_title("MST VR(X)", fontsize=15)
ax[1].set_title("MST VR(Z)", fontsize=15)
ax[2].set_title("MST superposed", fontsize=15)
plt.savefig(os.path.join(plots_dir,"MST_trees.png"))

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

def plot_hirarchical_clustering(point_cloud, ax):
    linkage_matrix = linkage(point_cloud, method='single', metric='euclidean')
    dendrogram(
        linkage_matrix,
        labels=[f"{i}" for i in range(len(point_cloud))],
        ax=ax,
        orientation='top'
    )
    ax.set_xlabel("Point Labels")
    ax.set_ylabel("Distance (Single linkage)")
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10,3))
plot_hirarchical_clustering(X, ax[0])
ax[0].set_title("Dendrogram for VR(X)", fontsize=13)
plot_hirarchical_clustering(Z, ax[1])
ax[1].set_title("Dendrogram for VR(Z)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir,"dendrogram_X_Z.png"))

In [ ]:
def plot_merge_tree(endpoints_0, reps_0, ax):
    max_x = np.max(endpoints_0)*1.1
    num_points = len(endpoints_0)+1
    y= np.linspace(0, 0.3*num_points, num_points)
    idx_death = []
    merging_into= []
    death_val = []
    for idx, (end, rep) in enumerate(zip(endpoints_0, reps_0)):
        ax.plot([0,end], [y[idx], y[idx]], c=mpl.colormaps["RdBu"](1/1.3), linewidth=3, zorder=0.5)
        idx_death.append(np.max(rep))
        merging_into.append(np.min(rep))
        death_val.append(end)
    
    # merge lines in red
    idx_death.append(0)
    for idx, (j, death) in enumerate(zip( merging_into, death_val)):
        death_merging = idx_death.index(j)
        ax.plot([death, death], [y[idx],y[death_merging]], linewidth=3, c=mpl.colormaps["RdBu"](0.3/1.3), zorder=0.5)

    xscale = ax.get_xlim()[1]-ax.get_xlim()[0]
    yscale = ax.get_ylim()[1]-ax.get_ylim()[0]
    for i, idx in enumerate(idx_death):
        ax.text(-0.015*xscale, y[i]-0.03*yscale, f"{idx}", zorder=0.7, fontsize=10, color="white", fontweight="bold")
        if i < len(idx_death)-1:
            death_x = endpoints_0[i]
            ax.text(death_x-0.015*xscale, y[i]-0.03*yscale, f"{merging_into[i]}", zorder=0.7, fontsize=10, color="white", fontweight="bold")

    ax.scatter(np.zeros(len(y)),y, s=100, marker="o", color=mpl.colormaps["RdBu"](1/1.3), zorder=0.6)
    ax.scatter(endpoints_0, y[:-1], s=100, marker="o", color=mpl.colormaps["RdBu"](0.3/1.3), zorder=0.6)
    ax.set_xlim(ax.get_xlim()[0]-0.1*xscale, ax.get_xlim()[1]+0.1*xscale)
    ax.set_ylim(ax.get_ylim()[0]-0.1*yscale, ax.get_ylim()[1]+0.1*yscale)
    # Top horizontal interval
    ax.plot([0,max_x*2], [y[-1],y[-1]], linewidth=3, c=mpl.colormaps["RdBu"](1/1.3), zorder=0.5)
    ax.set_yticks([])

In [ ]:
TMT_X_pairs = tdqual.compute_tmt_pairs(filt_X, pairs_arr_X)
TMT_Z_pairs = tdqual.compute_tmt_pairs(filt_Z, pairs_arr_Z)

In [ ]:
indices_X_Z = np.max(TMT_Z_pairs, axis=1)<X.shape[0]
TMT_X_Z_pairs = TMT_Z_pairs[indices_X_Z]
filt_X_Z = np.array(filt_Z)[indices_X_Z]
indices_X_Z = np.nonzero(indices_X_Z)[0]

In [ ]:
FX = tdqual.get_inclusion_matrix(TMT_X_pairs, TMT_X_Z_pairs) # Associated matrix

In [ ]:
FX_arr = np.zeros((X.shape[0]-1, X.shape[0]-1), dtype="int")
for i,col in enumerate(FX):
    FX_arr[:,i][col]=1

In [ ]:
FX_arr

In [ ]:
TMT_X_pairs

In [ ]:
TMT_X_Z_pairs

In [ ]:
filt_X_Z

In [ ]:
TMT_X_Z_pairs

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(ncols=2, figsize=(10,4))
plot_merge_tree(filt_X_Z, TMT_X_Z_pairs, ax[0])
ax[0].set_title("TMT$(X,Z)$", fontsize=20)

# Format strings exactly like before
row_labels = [f"({r[0]}, {val:1.2f}, {r[1]})" for r, val in zip(TMT_X_Z_pairs, filt_X_Z)]
col_labels = [f"({c[0]}, {val:1.2f}, {c[1]})" for c, val in zip(TMT_X_pairs, filt_X)]


sns.heatmap(
    FX_arr, 
    annot=True,               # Change to False if you don't want numbers inside the squares
    xticklabels=col_labels, 
    yticklabels=row_labels, 
    cmap='magma_r',
    cbar=False,
    annot_kws={"fontsize":20},
    ax=ax[1]
)

plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0, ha="right")
ax[1].set_title("F$(X,Z)$", fontsize=20)
ax[1].set_xlabel("TMT$(X)$", fontsize=20)
ax[1].set_ylabel("TMT$(X,Z)$", fontsize=20)

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "TMT_X_Z_inclusion_mat.png"))

In [ ]:
from tdqual.topological_data_quality_0 import add_columns_mod_2, get_pivot

def get_pivots_and_reduced_matrix(matrix_list, num_rows):
    """ Returns the pivots of a matrix given in list format"""
    pivots = []
    reduced_matrix = []
    pivot2column = np.full(num_rows, -1, dtype=int)
    for i, column in enumerate(matrix_list):
        reduce_column = list(column)
        piv = get_pivot(reduce_column)
        while(pivot2column[piv]>-1):
            reduce_column = add_columns_mod_2(reduce_column, reduced_matrix[pivot2column[piv]])
            piv = get_pivot(reduce_column)
            # we assume that columns are never reduced to the 0 column
        pivots.append(piv)
        reduced_matrix.append(reduce_column)
        pivot2column[piv] = i
    # end getting pivots
    return pivots, reduced_matrix

In [ ]:
pivots, reduced_matrix = get_pivots_and_reduced_matrix(FX, X.shape[0]-1)

In [ ]:
R_arr = np.zeros((X.shape[0]-1, X.shape[0]-1), dtype="int")
for i,col in enumerate(reduced_matrix):
    R_arr[:,i][col]=1

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10,4))
plot_merge_tree(filt_X_Z, TMT_X_Z_pairs, ax[0])
ax[0].set_title("TMT$(X,Z)$", fontsize=20)

# Format strings exactly like before
row_labels = [f"{val:1.2f}" for r, val in zip(TMT_X_Z_pairs, filt_X_Z)]
col_labels = [f"{val:1.2f}" for c, val in zip(TMT_X_pairs, filt_X)]

sns.heatmap(
    FX_arr, 
    annot=True,               # Change to False if you don't want numbers inside the squares
    xticklabels=col_labels, 
    yticklabels=row_labels, 
    cmap='magma_r',
    cbar=False,
    annot_kws={"fontsize":20},
    ax=ax[0]
)

plt.setp(ax[0].get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
plt.setp(ax[0].get_yticklabels(), rotation=0, ha="right", rotation_mode="anchor")
ax[0].set_title("F(X,Z)", fontsize=20)
ax[0].set_xlabel("B(X)", fontsize=20)
ax[0].set_ylabel("B(X,Z)", fontsize=20)

sns.heatmap(
    R_arr, 
    annot=True,               # Change to False if you don't want numbers inside the squares
    xticklabels=col_labels, 
    yticklabels=row_labels, 
    cmap='magma_r',
    cbar=False,
    annot_kws={"fontsize":20},
    ax=ax[1]
)

plt.setp(ax[1].get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
plt.setp(ax[1].get_yticklabels(), rotation=0, ha="right", rotation_mode="anchor")
ax[1].set_title("R(X,Z)", fontsize=20)
ax[1].set_xlabel("B(X)", fontsize=20)
ax[1].set_ylabel("B(X,Z)", fontsize=20)

plt.tight_layout()
plt.savefig(os.path.join(plots_dir, "R_X_Z.png"))

In [ ]:
fig, ax = plt.subplots(figsize=(3,3))
filt_X, filt_Z, matching = tdqual.compute_Mf_0(X, Z)
D_f, multiplicities = tdqual.compute_matching_diagram(filt_X, filt_Z, matching, _tol=1e-5)
tdqual.plot_matching_diagram(D_f, ax, size=30)
ax.set_xlim([-0.3,9])
plt.tight_layout()
plt.savefig(os.path.join(plots_dir,"matching_diagram_0.png"))

In [ ]:
D_f